In [ ]:
from frequent_itemsets import get_frequent_itemsets
from association_rules import get_association_rules
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment


frequent_itemsets = get_frequent_itemsets(min_sup=0.6)
association_rules = get_association_rules(frequent_itemsets, min_confidence=0.9)

frequent_itemsets['itemsets'] = frequent_itemsets['itemsets'].apply(lambda x: ', '.join(sorted(x)))
frequent_itemsets['support'] = frequent_itemsets['support'].round(4)

association_rules = association_rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']]
association_rules['antecedents'] = association_rules['antecedents'].apply(lambda x: ', '.join(sorted(x)))
association_rules['consequents'] = association_rules['consequents'].apply(lambda x: ', '.join(sorted(x)))

for col in ['support', 'confidence', 'lift']:
    if col in association_rules.columns:
        association_rules[col] = association_rules[col].round(4)


output_path = "./Output/subject_taken_together.xlsx"
with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    frequent_itemsets.to_excel(writer, sheet_name='Frequent Itemsets', index=False)
    association_rules.to_excel(writer, sheet_name='Association Rules', index=False)


wb = load_workbook(output_path)
for sheet_name in wb.sheetnames:
    ws = wb[sheet_name]
    for cell in ws[1]:
        cell.font = Font(bold=True)
        cell.alignment = Alignment(horizontal='center')

    for col in ws.columns:
        max_len = max(len(str(cell.value)) if cell.value else 0 for cell in col)
        ws.column_dimensions[col[0].column_letter].width = max_len + 2

wb.save(output_path)